In [ ]:
import os
import pandas as pd

# 定义文件路径
label_csv_path = '/home/b532root/data/b532zxy/AVEC/label.csv'  # 请替换为您的 label.csv 文件的实际路径
freeform_dir = '/home/b532root/data/b532zxy/AVEC/face/Freeform'

# 读取 label.csv 文件
label_df = pd.read_csv(label_csv_path)

# 获取实际存在的文件夹名称列表（排除 audio 文件夹）
existing_dirs = [name for name in os.listdir(freeform_dir) 
                 if os.path.isdir(os.path.join(freeform_dir, name)) and name != 'audio']

# 从 label.csv 中获取文件夹名称列表
label_files = label_df['file'].tolist()

# 找出不存在的文件夹名称
missing_files = [file for file in label_files if file not in existing_dirs]

# 输出不存在的文件夹记录
if missing_files:
    print("以下文件夹在指定目录中不存在：")
    missing_records = label_df[label_df['file'].isin(missing_files)]
    print(missing_records)
else:
    print("所有文件夹都存在于指定目录中。")


In [19]:
import os
import pandas as pd
import shutil
from sklearn.model_selection import train_test_split

# 设定路径
data_root = "/home/b532root/data/b532zxy/AVEC15"  # 你的原始数据存放目录
data_k = "/home/b532root/data/b532zxy/AVEC15"  # 目标存放新数据集的目录

# 读取 CSV 文件
label_csv_path = os.path.join(data_root, "all_label.csv")
df = pd.read_csv(label_csv_path)

# 获取所有文件名（不包含扩展名）和标签
X = df['file'].values  # 样本名称，如 '203_1'
y = df['label'].values  # 回归任务的数值

# 进行数据集划分 (Train 70%, Dev 15%, Test 15%)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, shuffle=True)
X_dev, X_test, y_dev, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, shuffle=True)

# 确保划分合理
print(f"Train: {len(X_train)}, Dev: {len(X_dev)}, Test: {len(X_test)}")

# 创建新目录
os.makedirs(data_k, exist_ok=True)
for new_split in ["train", "test", "dev"]:
    os.makedirs(os.path.join(data_k, new_split, "Freeform"), exist_ok=True)
    os.makedirs(os.path.join(data_k, new_split, "Northwind"), exist_ok=True)

import os
import shutil

def copy_samples(samples, src_root, dest_dir):
    """
    复制 Freeform 和 Northwind 的视频和音频数据
    """
    for sample in samples:
        found = False
        for split in ["train", "test", "dev"]:  # 在3个数据集文件夹中查找
            # 视频数据路径
            src_ff = os.path.join(src_root, split, "Freeform", sample)
            src_nw = os.path.join(src_root, split, "Northwind", sample)
            dest_ff = os.path.join(dest_dir, "Freeform", sample)
            dest_nw = os.path.join(dest_dir, "Northwind", sample)

            # 音频数据路径
            src_ff_audio = os.path.join(src_root, split, "Freeform", "audio", "frame_15", sample + ".npy")
            src_nw_audio = os.path.join(src_root, split, "Northwind", "audio", "frame_15", sample + ".npy")
            dest_ff_audio = os.path.join(dest_dir, "Freeform", "audio", "frame_15")
            dest_nw_audio = os.path.join(dest_dir, "Northwind", "audio", "frame_15")

            # 复制视频数据（如果存在）
            if os.path.exists(src_ff):  
                shutil.copytree(src_ff, dest_ff, dirs_exist_ok=True)
                found = True
            if os.path.exists(src_nw):  
                shutil.copytree(src_nw, dest_nw, dirs_exist_ok=True)
                found = True

            # 复制音频数据（如果存在）
            if os.path.exists(src_ff_audio):
                os.makedirs(dest_ff_audio, exist_ok=True)
                shutil.copy2(src_ff_audio, os.path.join(dest_ff_audio, sample + ".npy"))
            if os.path.exists(src_nw_audio):
                os.makedirs(dest_nw_audio, exist_ok=True)
                shutil.copy2(src_nw_audio, os.path.join(dest_nw_audio, sample + ".npy"))

        if not found:
            print(f"⚠️ 警告：样本 {sample} 在 train/test/dev 下都找不到，可能是数据丢失！")


# 执行数据复制
copy_samples(X_train, data_root, os.path.join(data_k, "train"))
copy_samples(X_dev, data_root, os.path.join(data_k, "dev"))
copy_samples(X_test, data_root, os.path.join(data_k, "test"))

# 生成新的 CSV
df_train = pd.DataFrame({"file": X_train, "label": y_train})
df_dev = pd.DataFrame({"file": X_dev, "label": y_dev})
df_test = pd.DataFrame({"file": X_test, "label": y_test})

df_train.to_csv(os.path.join(data_k, "train/train_label.csv"), index=False)
df_dev.to_csv(os.path.join(data_k, "dev/dev_label.csv"), index=False)
df_test.to_csv(os.path.join(data_k, "test/test_label.csv"), index=False)

print("✅ 数据集划分完成！")


Train: 93, Dev: 20, Test: 20
✅ 数据集划分完成！


In [1]:
import pandas as pd

# 读取 CSV 文件
train_df = pd.read_csv('/home/b532root/data/b532zxy/AVEC15/train/train_label.csv')
dev_df = pd.read_csv('/home/b532root/data/b532zxy/AVEC15/dev/dev_label.csv')
test_df = pd.read_csv('/home/b532root/data/b532zxy/AVEC15/test/test_label.csv')

# 假设标签列名为 "label"，如果不是请根据实际情况修改列名
print("训练集标签描述:")
print(train_df['label'].describe())

print("\n验证集标签描述:")
print(dev_df['label'].describe())

print("\n测试集标签描述:")
print(test_df['label'].describe())

训练集标签描述:
count    46.000000
mean     13.608696
std      11.490050
min       0.000000
25%       4.000000
50%      11.000000
75%      19.750000
max      41.000000
Name: label, dtype: float64

验证集标签描述:
count    41.000000
mean     13.707317
std      10.361573
min       0.000000
25%       5.000000
50%      12.000000
75%      23.000000
max      34.000000
Name: label, dtype: float64

测试集标签描述:
count    50.000000
mean     14.500000
std      11.597414
min       0.000000
25%       3.250000
50%      13.000000
75%      24.250000
max      43.000000
Name: label, dtype: float64


In [ ]:
import torch
import random

def compute_dataset_mean_std(dataset, sample_ratio=1.0):
    """
    计算给定数据集中所有 Freeform 图像的均值和标准差。
    
    参数：
        dataset: 一个 AudioVideoDataset 实例（假设 mode='train'，且使用了基础的 transforms，
                 不包含归一化操作）。
        sample_ratio: 采样比例（0~1之间），当数据集较大时可以只采样部分样本进行估计。
    
    返回：
        mean: Tensor，形状为 [3]，每个通道的均值。
        std: Tensor，形状为 [3]，每个通道的标准差。
    """
    total_sum = torch.zeros(3)
    total_sum_sq = torch.zeros(3)
    total_pixels = 0

    # 获取所有索引；如果 sample_ratio 小于1，则随机采样部分样本进行计算
    n = len(dataset)
    indices = list(range(n))
    if sample_ratio < 1.0:
        indices = random.sample(indices, int(n * sample_ratio))

    for idx in indices:
        sample = dataset[idx]
        # 使用的是 Freeform 图片，返回的 'ff_video_features' 的形状为 [select_frames, C, H, W]
        imgs = sample['ff_video_features']  # Tensor，形状例如 (15, 3, 256, 256)
        # 获取形状信息
        frames, channels, height, width = imgs.shape
        num_pixels = frames * height * width

        # 累计每个通道的像素和与平方和
        # imgs.sum(dim=[0,2,3]) 的形状为 [C]，即对所有帧和空间维度求和
        total_sum += imgs.sum(dim=[0, 2, 3])
        total_sum_sq += (imgs ** 2).sum(dim=[0, 2, 3])
        total_pixels += num_pixels

    mean = total_sum / total_pixels
    std = torch.sqrt(total_sum_sq / total_pixels - mean ** 2)
    return mean, std


if __name__ == '__main__':
    # 假设你的训练数据路径和标签路径如下（根据你的配置修改）
    train_data_path = '/home/b532root/data/b532zxy/AVEC15/train'  # 示例路径
    train_label_path = '/home/b532root/data/b532zxy/AVEC15/train/train_label.csv'
    dev_data_path = '/home/b532root/data/b532zxy/AVEC15/dev'  # 示例路径
    dev_label_path = '/home/b532root/data/b532zxy/AVEC15/dev/dev_label.csv'
    test_data_path = '/home/b532root/data/b532zxy/AVEC15/test'  # 示例路径
    test_label_path = '/home/b532root/data/b532zxy/AVEC15/test/test_label.csv'
    
    # 构造 AudioVideoDataset，此处 mode 设置为 'train'
    from dataloader import AudioVideoDataset
    train_dataset = AudioVideoDataset(av_path=train_data_path, label_path=train_label_path, mode='train', num_frames=15)
    dev_dataset = AudioVideoDataset(av_path=dev_data_path, label_path=dev_label_path, mode='eval', num_frames=15)
    test_dataset = AudioVideoDataset(av_path=test_data_path, label_path=test_label_path, mode='test', num_frames=15)
    
    # 计算均值和标准差（你可以采样全部或者部分数据，sample_ratio=0.5 表示只用 50% 样本）
    train_mean, train_std = compute_dataset_mean_std(train_dataset, sample_ratio=1.0)
    dev_mean, dev_std = compute_dataset_mean_std(dev_dataset, sample_ratio=1.0)
    test_mean, test_std = compute_dataset_mean_std(test_dataset, sample_ratio=1.0)
    print("计算得到的均值:", train_mean)
    print("计算得到的标准差:", train_std)
    print("计算得到的均值:", dev_mean)
    print("计算得到的标准差:", dev_std)
    print("计算得到的均值:", test_mean)
    print("计算得到的标准差:", test_std)

FileNotFoundError: [Errno 2] No such file or directory: '/home/b532root/data/b532zxy/AVECK15/train/Northwind/audio/frame_15/203_1.npy'

In [9]:
import torch
import random
import numpy as np
from tqdm import tqdm  # 添加进度条

def compute_audio_mean_std(dataset, sample_ratio=1.0):
    """
    计算给定数据集中所有音频特征的均值和标准差
    
    参数：
        dataset: AudioVideoDataset 实例
        sample_ratio: 采样比例 (0~1)
    
    返回：
        mean: numpy数组，形状 [feature_dim]
        std: numpy数组，形状 [feature_dim]
    """
    # 初始化累计变量
    total_sum = None
    total_sum_sq = None
    total_count = 0

    # 获取采样索引
    n = len(dataset)
    indices = list(range(n))
    if sample_ratio < 1.0:
        indices = random.sample(indices, int(n * sample_ratio))

    # 遍历数据集
    for idx in tqdm(indices, desc="计算音频统计量"):
        sample = dataset[idx]
        
        # 处理 Freeform 和 Northwind 两个音频特征
        for audio_feat in [sample['ff_audio_features'], sample['nw_audio_features']]:
            # 确保是 numpy 数组
            if isinstance(audio_feat, torch.Tensor):
                audio_feat = audio_feat.numpy()
            
            # 初始化累计变量
            if total_sum is None:
                feature_dim = audio_feat.shape[-1]
                total_sum = np.zeros(feature_dim)
                total_sum_sq = np.zeros(feature_dim)
            
            # 沿时间维度求和 (假设音频特征形状为 [时间步, 特征维度])
            sum_per_feature = audio_feat.sum(axis=0)
            sum_sq_per_feature = (audio_feat ** 2).sum(axis=0)
            
            # 累计统计量
            total_sum += sum_per_feature
            total_sum_sq += sum_sq_per_feature
            total_count += audio_feat.shape[0]  # 时间步数

    # 计算最终统计量
    mean = total_sum / total_count
    std = np.sqrt(total_sum_sq / total_count - mean ** 2)
    return mean, std

if __name__ == '__main__':
    # 原有视频统计代码...
    train_data_path = '/home/b532root/data/b532zxy/AVEC15/train'  # 示例路径
    train_label_path = '/home/b532root/data/b532zxy/AVEC15/train/train_label.csv'
    dev_data_path = '/home/b532root/data/b532zxy/AVEC15/dev'  # 示例路径
    dev_label_path = '/home/b532root/data/b532zxy/AVEC15/dev/dev_label.csv'
    test_data_path = '/home/b532root/data/b532zxy/AVEC15/test'  # 示例路径
    test_label_path = '/home/b532root/data/b532zxy/AVEC15/test/test_label.csv'
    
    # 构造 AudioVideoDataset，此处 mode 设置为 'train'
    from dataloader import AudioVideoDataset  # 替换为实际模块名
    train_dataset = AudioVideoDataset(av_path=train_data_path, label_path=train_label_path, mode='train', num_frames=15)
    dev_dataset = AudioVideoDataset(av_path=dev_data_path, label_path=dev_label_path, mode='eval', num_frames=15)
    test_dataset = AudioVideoDataset(av_path=test_data_path, label_path=test_label_path, mode='test', num_frames=15)

    # 新增音频统计计算
    print("\n正在计算音频统计量...")
    train_audio_mean, train_audio_std = compute_audio_mean_std(train_dataset)
    dev_audio_mean, dev_audio_std = compute_audio_mean_std(dev_dataset)
    test_audio_mean, test_audio_std = compute_audio_mean_std(test_dataset)
    
    print("\n训练集音频统计:")
    print(f"Mean: {train_audio_mean.round(4)}")
    print(f"Std: {train_audio_std.round(4)}")
    
    print("\n验证集音频统计:")
    print(f"Mean: {dev_audio_mean.round(4)}")
    print(f"Std: {dev_audio_std.round(4)}")
    
    print("\n测试集音频统计:")
    print(f"Mean: {test_audio_mean.round(4)}")
    print(f"Std: {test_audio_std.round(4)}")


正在计算音频统计量...


计算音频统计量: 100%|██████████| 46/46 [00:01<00:00, 25.36it/s]


训练集音频统计:
Mean: [ 0.  0. -0.  0.  0.  0. -0. -0. -0.  0. -0.  0. -0.  0.  0. -0.  0. -0.
  0. -0.  0.  0. -0.  0. -0. -0.  0.  0. -0.  0.  0. -0.  0.  0.  0.  0.
  0.  0.  0.  0.  0.  0. -0.  0.  0. -0.  0. -0. -0.  0. -0.  0.  0. -0.
  0. -0.  0. -0. -0.  0.  0. -0.  0. -0.  0.  0. -0. -0.  0. -0. -0. -0.
  0.  0.  0. -0. -0.  0.  0.  0. -0.  0.  0. -0.  0. -0. -0.  0.  0.  0.
 -0. -0.  0.  0.  0. -0. -0.  0. -0.  0.  0.  0. -0. -0.  0. -0. -0. -0.
  0. -0. -0. -0. -0. -0. -0.  0. -0.  0. -0. -0.  0. -0. -0. -0. -0. -0.
  0.  0.]
Std: [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1.]

验证集音频统计:
Mean: [-9.6500e-02 -1.1860e-01 -5.6600e-02  1.0000e-01 -1

In [ ]:
import torch
import torch.nn as nn

class AdaptiveWingLoss(nn.Module):
    def __init__(self, alpha=2.1, omega=14, epsilon=1, theta=0.5):
        super(AdaptiveWingLoss, self).__init__()
        self.alpha = alpha
        self.omega = omega
        self.epsilon = epsilon
        self.theta = theta

    def forward(self, pred, target):
        # 计算误差
        delta = torch.abs(target - pred)
        
        # 根据误差与 theta 的比较来选择合适的公式
        loss = torch.where(delta < self.theta,
                           self.omega * torch.log(1 + (delta / self.epsilon) ** self.alpha),
                           self.omega * (delta - self.theta))
        
        return torch.mean(loss)

# 使用示例
# 假设 pred 是模型输出的热力图，target 是加载的真实热力图标签
# pred 和 target 的形状为 [batch_size, 50, 68, 64, 64]
loss_fn = AdaptiveWingLoss()
pred = torch.randn(2, 50, 68, 64, 64)  # 模型预测输出
target = torch.randn(2, 50, 68, 64, 64)  # 热力图标签

# 计算损失，遍历50张图片的损失并求平均
batch_size, num_frames, num_landmarks, height, width = pred.shape
total_loss = 0

for i in range(num_frames):
    frame_loss = loss_fn(pred[:, i], target[:, i])
    total_loss += frame_loss

average_loss = total_loss / num_frames
print("Average Loss over 50 frames:", average_loss.item())
